# ToothFairy3 → Sol/Sağ IAC — nnU-Net Colab Pipeline
Bu notebook Colab'da **prototipleme ve smoke-test** içindir. Tam 3D eğitim uzun
sürer ve Colab oturumu koparsa devam edebilmek için **her şey Google Drive'a**
yazdırılır (`--c` ile resume). Runtime: **GPU** seçin (Runtime > Change runtime type).

## 1) Drive'ı bağla ve klasör yapısını kur

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/toothfairy_iac'   # kalıcı kök
os.makedirs(ROOT, exist_ok=True)
os.environ['nnUNet_raw']          = f'{ROOT}/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = f'{ROOT}/nnUNet_preprocessed'
os.environ['nnUNet_results']      = f'{ROOT}/nnUNet_results'
for p in ['nnUNet_raw','nnUNet_preprocessed','nnUNet_results']:
    os.makedirs(os.environ[p], exist_ok=True)
print('Env hazır:', {k:os.environ[k] for k in ['nnUNet_raw','nnUNet_preprocessed','nnUNet_results']})
!nvidia-smi -L

## 2) nnU-Net v2 ve bağımlılıkları kur

In [ ]:
!pip -q install nnunetv2 nibabel tqdm SimpleITK
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 3) Custom trainer'ı nnU-Net içine kopyala
`nnUNetTrainerIAC.py` dosyasını önce Drive'daki `toothfairy_iac/` köküne yükleyin.

In [ ]:
import nnunetv2, shutil, os
dst_dir = os.path.join(os.path.dirname(nnunetv2.__file__),
                       'training','nnUNetTrainer','variants','data_augmentation')
shutil.copy(f'{ROOT}/nnUNetTrainerIAC.py', dst_dir)
print('Trainer kopyalandı ->', dst_dir)

## 4) Veriyi hazırla (TF3 → 3-sınıflı IAC)
ToothFairy3'ü Drive'a açtığınızı ve `imagesTr` / `labelsTr` yollarını
aşağıda ayarladığınızı varsayar. Önce sadece **etiket doğrulama** çalıştırın.

In [ ]:
# prepare_toothfairy3_iac.py dosyasını da ROOT'a yükleyin
TF3_IMAGES = f'{ROOT}/ToothFairy3/imagesTr'
TF3_LABELS = f'{ROOT}/ToothFairy3/labelsTr'

# 4a) Sadece etiket id'lerini doğrula (3=Left, 4=Right mi?)
!python {ROOT}/prepare_toothfairy3_iac.py --tf3_labels "{TF3_LABELS}" --verify_only

In [ ]:
# 4b) Doğruladıysanız dönüştür (symlink disk tasarrufu sağlar)
!python {ROOT}/prepare_toothfairy3_iac.py \
    --tf3_images "{TF3_IMAGES}" --tf3_labels "{TF3_LABELS}" \
    --out_root "$nnUNet_raw" --dataset_id 111 --dataset_name IAC_LR --symlink

## 5) Planlama + preprocessing

In [ ]:
!nnUNetv2_plan_and_preprocess -d 111 --verify_dataset_integrity

## 6) Smoke-test eğitimi (50 epoch)
Pipeline'ın uçtan uca çalıştığını doğrulamak için. Gerçek sonuç beklemeyin.

In [ ]:
!nnUNetv2_train 111 3d_fullres 0 -tr nnUNetTrainerIAC_NoMirror_50ep --npz

## 7) Tam eğitim + resume
Colab koparsa aynı hücreyi `--c` ile tekrar çalıştırın; Drive'daki
checkpoint'ten devam eder. Tek fold uzun sürer; 3d_lowres ile iterasyon
daha hızlıdır.

In [ ]:
# ilk başlatma:
!nnUNetv2_train 111 3d_fullres 0 -tr nnUNetTrainerIAC_NoMirror --npz
# kopma sonrası devam:
# !nnUNetv2_train 111 3d_fullres 0 -tr nnUNetTrainerIAC_NoMirror --npz --c

## Notlar
- **Bellek darsa:** `3d_lowres` konfigürasyonu ya da `-p nnUNetPlans` içinde patch/batch küçültme.
- **5-fold:** fold 0..4 ayrı ayrı; Colab'da genelde 1 fold + Drive resume pratik.
- **Alternatif:** Kaggle (haftalık 30h, daha stabil) veya kiralık GPU tam eğitim için daha uygun.